## Checking our current MetaStore

In [0]:
# Use the value without ever printing it


In [0]:
#  To masks the input of the values below
import ipywidgets as widgets

app_id = widgets.Password(description="App ID:")
storage_account = widgets.Password(description="Storage Account:")
dir_id = widgets.Password(description="Dir ID:")
client_secret = widgets.Password(description="Client Secret:")

display(app_id, storage_account, dir_id, client_secret)

In [0]:
# Use the value without ever printing it
secret_value = client_secret.value


In [0]:
%sql
SELECT CURRENT_METASTORE();


### Checking for Catalog

In [0]:
%sql
SHOW CATALOGS;


### Connection Credentials

In [0]:
# COMMAND ----------
# DBTITLE 1,Secure Connection via Widgets
# 1. Create the input boxes (Widgets) at the top of the screen
dbutils.widgets.text("app_id",          "", "Azure App ID")
dbutils.widgets.text("dir_id",          "", "Azure Directory ID")
dbutils.widgets.text("secret",          "", "Azure Client Secret")
dbutils.widgets.text("storage_account", "retailstorageaccountdon.dfs.core.windows.net", "Storage Account")

# 2. Read the values you type into those boxes
app_id          = dbutils.widgets.get("app_id")
dir_id          = dbutils.widgets.get("dir_id")
secret          = dbutils.widgets.get("secret")
storage_account = dbutils.widgets.get("storage_account")

# 3. Apply the Spark Configuration using the variables
if app_id and secret:
    spark.conf.set(f"fs.azure.account.auth.type.{storage_account}", "OAuth")
    spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
    spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}", app_id)
    spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}", secret)
    spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}", f"https://microsoftonline.com{dir_id}/oauth2/token")
    print("Authentication configured via Widgets.")
else:
    print("Please enter credentials in the widgets at the top of the page.")


In [0]:
spark.conf.set(f"fs.azure.account.auth.type.{storage_account}", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}", app_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}", secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}", f"https://login.microsoftonline.com/{dir_id}/oauth2/token")



### Accessing the path to our data

In [0]:
# Accessing the correct path: container=retail, then navigate into bronze folder
dbutils.fs.ls('abfss://retail@retailstorageaccountdon.dfs.core.windows.net/bronze/')

In [0]:
# Accessing the store path and giving it a variable name so we don't have to repeat it every time.
base_path = 'abfss://retail@retailstorageaccountdon.dfs.core.windows.net/bronze'

In [0]:
# To see exactly what files are in that folder
dbutils.fs.ls(f'{base_path}/customer/Donatus-dataProjects/azure-data-engineer/refs/heads/main/')

## Reading the Data

In [0]:
# Reading customer parquet
df_customers = spark.read.parquet(f'{base_path}/customer/Donatus-dataProjects/azure-data-engineer/refs/heads/main/customers.parquet')

# Display
display(df_customers)

In [0]:
# ---- Product ----
df_products = spark.read.parquet(f'{base_path}/product/dbo.products.parquet')
df_products.display()

In [0]:
# ---- Store ----
df_stores = spark.read.parquet(f'{base_path}/store/dbo.stores.parquet')
df_stores.display()

In [0]:
# ---- Transaction (note the typo in folder name) ----
df_transactions = spark.read.parquet(f'{base_path}/transacrion/dbo.transactions.parquet')
df_transaction.display()

In [0]:
## importing all the functions 
from pyspark.sql.functions import *
##importing all the types
from pyspark.sql.types import *


## DATA CLEANING

In [0]:
from pyspark.sql.functions import col
# Convert types and clean data
df_transactions = df_transactions.select(
    col("transaction_id").cast("int"),
    col("customer_id").cast("int"),
    col("product_id").cast("int"),
    col("store_id").cast("int"),
    col("quantity").cast("int"),
    col("transaction_date").cast("date")
)

df_products = df_products.select(
    col("product_id").cast("int"),
    col("product_name"),
    col("category"),
    col("price").cast("double")
)

df_stores = df_stores.select(
    col("store_id").cast("int"),
    col("store_name"),
    col("location")
)

df_customers = df_customers.select(
    "customer_id", "first_name", "last_name", "email", "city", "registration_date"
).dropDuplicates(["customer_id"])

In [0]:
# Previewing all
for name, df in [("Customer", df_customers), ("Product", df_products), 
                 ("Store", df_stores), ("Transaction", df_transactions)]:
    print(f"\n{'='*30}\n {name} | Rows: {df.count()} | Cols: {len(df.columns)}")
    display(df)

In [0]:
# Join all data
df_silver = df_transactions \
    .join(df_customers, "customer_id") \
    .join(df_products, "product_id") \
    .join(df_stores, "store_id") \
    .withColumn("total_amount", col("quantity") * col("price"))

## JOINING THE TABLES

In [0]:
 display(df_silver)

## DUMPING IT INTO THE SILVER LAYER

In [0]:
# Save the joined silver table to ADLS
silver_path = 'abfss://retail@retailstorageaccountdon.dfs.core.windows.net/silver/'

df_silver.write.mode("overwrite").format("delta").save(silver_path)

print("✅ Silver table saved to ADLS!")

### Creating a Delta Table on top of the saved silver data so it can be queried using SQL. 

In [0]:
# Creating Delta table pointing to my silver path
spark.sql(f"""
CREATE TABLE IF NOT EXISTS retail_silver
USING DELTA
LOCATION 'abfss://retail@retailstorageaccountdon.dfs.core.windows.net/silver/'
""")


In [0]:
%sql
SELECT *
FROM retail_silver
LIMIT 10

I CAN USE SQL OR SPARK TO READ THE DATA

In [0]:
# Now query it with SPARKSQL
# spark.sql("SELECT * FROM retail_silver").display()
spark.sql("SELECT * FROM retail_silver LIMIT 10").display()

In [0]:
df_silver = spark.read.format("delta").load('abfss://retail@retailstorageaccountdon.dfs.core.windows.net/silver/')

# GOLD LAYER

In [0]:
from pyspark.sql.functions import sum, col, month, year, round, concat, lit, first


# 2. Group by Year, Month, and IDs to keep the data clean and impactful
df_gold = df_silver_prepared.groupBy(
   "transaction_date",
    "product_id",
    "store_id",
    "customer_id",
    "product_name",
    "category",
    "store_name",
    "location"
).agg(
    # Descriptive Information (Enriched Customer Details)
    concat(first("first_name"), lit(" "), first("last_name")).alias("customer_full_name"),
    first("city").alias("customer_city"),
    
    # Keeping descriptive labels for the group
    first("product_name").alias("display_product_name"),
    first("category").alias("display_category"),
    first("store_name").alias("display_store_name"),
    first("location").alias("display_store_location"),
    
    # Primary Sales Metrics
    sum("quantity").alias("total_quantity_sold"),
    round(sum("total_amount"), 2).alias("total_sales_amount"),
    countDistinct("transaction_id").alias("transaction_volume")
)



In [0]:

# Display to see the new tenure column
display(df_gold)

## DUMPING IT INTO THE GOLD LAYER

In [0]:
# Save the the gold enriched table to the gold table to ADLS
gold_path = 'abfss://retail@retailstorageaccountdon.dfs.core.windows.net/gold/'

df_gold.write.mode("overwrite").format("delta").save(gold_path)

print("✅ Silver table saved to ADLS!")

### Creaing a Delta Table on top of the saved gold data so it can be queried using SQL

In [0]:
# Creating Delta table pointing to my silver path
spark.sql(f"""
CREATE TABLE IF NOT EXISTS retail_gold
USING DELTA
LOCATION 'abfss://retail@retailstorageaccountdon.dfs.core.windows.net/gold/'
""")


In [0]:
%sql
SELECT *
FROM retail_gold
LIMIT 10

#### I CAN USE SQL OR SPARK TO READ THE DATA

In [0]:
# Now query it with SPARKSQL
spark.sql("SELECT * FROM retail_gold LIMIT 10").display()

In [0]:
%sql
SHOW TABLES IN retailazuredatabricks.default;


In [0]:
%sql
--table to allow column updates
ALTER TABLE retailazuredatabricks.default.retail_gold SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name');


In [0]:
%sql

-- Force the metastore to re-scan the files for the new column
REFRESH TABLE retailazuredatabricks.default.retail_gold;


In [0]:
df_gold.printSchema()
df_gold.show(5)